# <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;  " > Practice lab: Deep Learning for Content-Based Filtering

In this exercise, you will implement content-based filtering using a neural network to build a recommender system for movies. 

# Outline <img align="left" src="./images/film_reel.png"     style=" width:40px;  " >
- [ 1 - Packages](#1)
- [ 2 - Movie ratings dataset](#2)
  - [ 2.1 Content-based filtering with a neural network](#2.1)
  - [ 2.2 Preparing the training data](#2.2)
- [ 3 - Neural Network for content-based filtering](#3)
  - [ 3.1 Predictions](#3.1)
    - [ Exercise 1](#ex01)
- [ 4 - Congratulations!](#4)


# 实践实验：用于基于内容过滤的深度学习

在本练习中，你将使用神经网络实现基于内容的过滤，以构建电影推荐系统。

# 大纲
- [ 1 - 软件包](#1)
- [ 2 - 电影评分数据集](#2)
  - [ 2.1 使用神经网络进行基于内容的过滤](#2.1)
  - [ 2.2 准备训练数据](#2.2)
- [ 3 - 用于基于内容过滤的神经网络](#3)
  - [ 3.1 预测](#3.1)
    - [ 练习 1](#ex01)
- [ 4 - 恭喜！](#4)

<a name="1"></a>
## 1 - Packages <img align="left" src="./images/movie_camera.png"     style=" width:40px;  ">
We will use familiar packages, NumPy, TensorFlow and helpful routines from [scikit-learn](https://scikit-learn.org/stable/). We will also use [tabulate](https://pypi.org/project/tabulate/) to neatly print tables and [Pandas](https://pandas.pydata.org/) to organize tabular data.

## 1 - 软件包
我们将使用熟悉的软件包：NumPy、TensorFlow 以及来自 [scikit-learn](https://scikit-learn.org/stable/) 的实用工具。我们还将使用 [tabulate](https://pypi.org/project/tabulate/) 来整洁地打印表格，并使用 [Pandas](https://pandas.pydata.org/) 来组织表格数据。

In [ ]:
import numpy as np
import numpy.ma as ma
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - Movie ratings dataset <img align="left" src="./images/film_rating.png" style=" width:40px;" >
The data set is derived from the [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) dataset. 

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

The original dataset has 9000 movies rated by 600 users with ratings on a scale of 0.5 to 5 in 0.5 step increments. The dataset has been reduced in size to focus on movies from the years since 2000 and popular genres. The reduced dataset has $n_u = 395$ users and $n_m= 694$ movies. For each movie, the dataset provides a movie title, release date, and one or more genres. For example "Toy Story 3" was released in 2010 and has several genres: "Adventure|Animation|Children|Comedy|Fantasy|IMAX".  This dataset contains little information about users other than their ratings. This dataset is used to create training vectors for the neural networks described below. 

## 2 - 电影评分数据集
该数据集源自 [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) 数据集。

[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集包含 600 位用户对 9000 部电影的评分，评分范围为 0.5 到 5 分，步长为 0.5。为了专注于 2000 年以来的电影和热门类型，该数据集的规模已被缩减。缩减后的数据集有 $n_u = 395$ 位用户和 $n_m= 694$ 部电影。对于每部电影，数据集提供了电影名称、上映日期和一个或多个类型。例如，“Toy Story 3”（《玩具总动员 3》）上映于 2010 年，包含多个类型：“Adventure|Animation|Children|Comedy|Fantasy|IMAX”。除了评分之外，该数据集几乎不包含关于用户的其他信息。该数据集将用于为下面描述的神经网络创建训练向量。

<a name="2.1"></a>
### 2.1 Content-based filtering with a neural network

In the collaborative filtering lab, you generated two vectors, a user vector and an item/movie vector whose dot product would predict a rating. The vectors were derived solely from the ratings.   

Content-based filtering also generates a user and movie feature vector but recognizes there may be other information available about the user and/or movie that may improve the prediction. The additional information is provided to a neural network which then generates the user and movie vector as shown below.
<figure>
    <center> <img src="./images/RecSysNN.png"   style="width:500px;height:280px;" ></center>
</figure>
The movie content provided to the network is a combination of the original data and some 'engineered features'. Recall the feature engineering discussion and lab from Course 1, Week 2, lab 4. The original features are the year the movie was released and the movie's genre presented as a one-hot vector. There are 14 genres. The engineered feature is an average rating derived from the user ratings. Movies with multiple genre have a training vector per genre. 

The user content is composed of only engineered features. A per genre average rating is computed per user. Additionally, a user id, rating count and rating average are available, but are not included in the training or prediction content. They are useful in interpreting data.

The training set consists of all the ratings made by the users in the data set. The user and movie/item vectors are presented to the above network together as a training set. The user vector is the same for all the movies rated by the user. 

Below, let's load and display some of the data.

### 2.1 使用神经网络进行基于内容的过滤

在协同过滤实验中，你生成了两个向量——用户向量和物品/电影向量，它们的点积用于预测评分。这些向量完全是从评分中推导出来的。

基于内容的过滤同样会生成用户特征向量和电影特征向量，但它认识到可能存在关于用户和/或电影的其他信息，这些信息可以改进预测。这些额外信息被提供给神经网络，然后由神经网络生成用户向量和电影向量，如上图所示。

提供给网络的电影内容是原始数据和一些“工程特征”的组合。回顾课程 1 第 2 周实验 4 中关于特征工程的讨论和实验。原始特征是电影的上映年份以及以独热（one-hot）向量形式表示的电影类型。共有 14 种类型。工程特征是从用户评分中计算出的平均评分。具有多种类型的电影会为每种类型生成一个训练向量。

用户内容完全由工程特征组成，即按用户计算每种类型的平均评分。此外，用户 ID、评分次数和平均评分也是可用的，但它们不包含在训练或预测内容中，而是用于解释数据。

训练集由数据集中用户给出的所有评分组成。用户向量和电影/物品向量一起作为训练集输入到上述网络中。对于该用户评分的所有电影，用户向量都是相同的。

下面，让我们加载并显示部分数据。

In [ ]:
# Load Data, set configuration variables
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items
scaledata = True  # applies the standard scalar to data if true
print(f"Number of training vectors: {len(item_train)}")

Some of the user and item/movie features are not used in training. Below, the features in brackets "[]" such as the "user id", "rating count" and "rating ave" are not included when the model is trained and used. Note, the user vector is the same for all the movies rated.

部分用户特征和物品/电影特征不用于训练。在下面，方括号“[]”中的特征（如“user id”、“rating count”和“rating ave”）在训练和使用模型时不包含在内。注意，对于该用户评分的所有电影，用户向量都是相同的。

In [ ]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

In [ ]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

In [ ]:
print(f"y_train[:5]: {y_train[:5]}")

Above, we can see that movie 6874 is an action movie released in 2003. User 2 rates action movies as 3.9 on average. Further, movie 6874 was also listed in the Crime and Thriller genre. MovieLens users gave the movie an average rating of 4. A training example consists of a row from both tables and a rating from y_train.

在上面，我们可以看到电影 6874 是一部 2003 年上映的动作片。用户 2 对动作片的平均评分为 3.9。此外，电影 6874 还被列为犯罪片和惊悚片。MovieLens 用户给这部电影的平均评分为 4。一个训练样本由两个表中的各一行以及 `y_train` 中的一个评分组成。

<a name="2.2"></a>
### 2.2 Preparing the training data
Recall in Course 1, Week 2, you explored feature scaling as a means of improving convergence. We'll scale the input features using the [scikit learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html). This was used in Course 1, Week 2, Lab 5.  Below, the inverse_transform is also shown to produce the original inputs.

### 2.2 准备训练数据
回顾在课程 1 第 2 周中，你探索了特征缩放作为改善收敛性的方法。我们将使用 [scikit-learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) 对输入特征进行缩放，这在课程 1 第 2 周实验 5 中使用过。下面还展示了使用 `inverse_transform` 来还原原始输入。

In [ ]:
# scale training data
if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    scalerItem = StandardScaler()
    scalerItem.fit(item_train)
    item_train = scalerItem.transform(item_train)

    scalerUser = StandardScaler()
    scalerUser.fit(user_train)
    user_train = scalerUser.transform(user_train)

    print(np.allclose(item_train_save, scalerItem.inverse_transform(item_train)))
    print(np.allclose(user_train_save, scalerUser.inverse_transform(user_train)))

To allow us to evaluate the results, we will split the data into training and test sets as was discussed in Course 2, Week 3. Here we will use [sklean train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to split and shuffle the data. Note that setting the initial random state to the same value ensures item, user, and y are shuffled identically.

为了评估结果，我们将按照课程 2 第 3 周讨论的方法，将数据划分为训练集和测试集。这里我们将使用 [sklearn train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来划分和打乱数据。请注意，将初始随机状态（random state）设置为相同的值可以确保物品、用户和 `y` 以完全相同的方式被打乱。

In [ ]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

The scaled, shuffled data now has a mean of zero.

经过缩放和打乱的数据现在均值为零。

In [ ]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

Scale the target ratings using a Min Max Scaler to scale the target to be between -1 and 1. We use scikit-learn because it has an inverse_transform. [scikit learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

使用 MinMaxScaler 对目标评分进行缩放，将目标值缩放到 -1 和 1 之间。我们使用 scikit-learn 是因为它提供了 `inverse_transform` 方法。[scikit-learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)

In [ ]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train = scaler.transform(y_train.reshape(-1, 1))
ynorm_test = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train.shape, ynorm_test.shape)

<a name="3"></a>
## 3 - Neural Network for content-based filtering
Now, let's construct a neural network as described in the figure above. It will have two networks that are combined by a dot product. You will construct the two networks. In this example, they will be identical. Note that these networks do not need to be the same. If the user content was substantially larger than the movie content, you might elect to increase the complexity of the user network relative to the movie network. In this case, the content is similar, so the networks are the same.

- Use a Keras sequential model
    - The first layer is a dense layer with 256 units and a relu activation.
    - The second layer is a dense layer with 128 units and a relu activation.
    - The third layer is a dense layer with `num_outputs` units and a linear or no activation.   
    
The remainder of the network will be provided. The provided code does not use the Keras sequential model but instead uses the Keras [functional api](https://keras.io/guides/functional_api/). This format allows for more flexibility in how components are interconnected.


## 3 - 用于基于内容过滤的神经网络
现在，让我们按照上图所述构建一个神经网络。它将包含两个通过点积组合的网络。你将构建这两个网络。在本例中，它们是完全相同的。请注意，这两个网络并不一定需要相同。如果用户内容比电影内容庞大得多，你可以选择相对电影网络增加用户网络的复杂度。在本例中，两者的内容相似，因此网络结构相同。

- 使用 Keras Sequential 模型
    - 第一层是具有 256 个单元和 `relu` 激活函数的全连接层（Dense）。
    - 第二层是具有 128 个单元和 `relu` 激活函数的全连接层（Dense）。
    - 第三层是具有 `num_outputs` 个单元和线性（或无）激活函数的全连接层（Dense）。   
    
网络的其余部分已为你提供。所提供的代码没有使用 Keras Sequential 模型，而是使用了 Keras [函数式 API（Functional API）](https://keras.io/guides/functional_api/)。这种形式在组件如何相互连接方面提供了更大的灵活性。

In [ ]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###   
      
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     

    ### END CODE HERE ###  
])

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# create the item input and point to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# specify the inputs and output of the model
model = Model([input_user, input_item], output)

model.summary()

In [ ]:
# Public tests
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
  You can create a dense layer with a relu activation as shown.
    
```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b> Click for solution</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```
</details>
</details>

    


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  你可以如下所示创建一个带有 relu 激活函数的全连接层。
    
```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b>点击查看解答</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```
</details>
</details>

We'll use a mean squared error loss and an Adam optimizer.

我们将使用均方误差（mean squared error）损失和 Adam 优化器。

In [ ]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [ ]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], ynorm_train, epochs=30)

Evaluate the model to determine loss on the test data. It is comparable to the training loss indicating the model has not substantially overfit the training data.

评估模型以确定在测试数据上的损失。它与训练损失相当，表明模型没有对训练数据产生明显的过拟合。

In [ ]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], ynorm_test)

<a name="3.1"></a>
### 3.1 Predictions
Below, you'll use your model to make predictions in a number of circumstances. 
#### Predictions for a new user
First, we'll create a new user and have the model suggest movies for that user. After you have tried this example on the example user content, feel free to change the user content to match your own preferences and see what the model suggests. Note that ratings are between 0.5 and 5.0, inclusive, in half-step increments.

### 3.1 预测
下面，你将在多种场景下使用你的模型进行预测。
#### 为新用户进行预测
首先，我们将创建一个新用户，并让模型为该用户推荐电影。在尝试了示例用户内容之后，你可以自由修改用户内容以匹配你自己的偏好，看看模型会推荐什么。注意，评分范围为 0.5 到 5.0（含），以 0.5 为步长。

In [ ]:
new_user_id = 5000
new_rating_ave = 1.0
new_action = 1.0
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 5
new_crime = 1
new_documentary = 1
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 5
new_scifi = 5
new_thriller = 1
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])


Let's look at the top-rated movies for the new user. Recall, the user vector had genres that favored Comedy and Romance.
Below, we'll use a set of movie/item vectors, `item_vecs` that have a vector for each movie in the training/test set. This is matched with the user vector above and the scaled vectors are used to predict ratings for all the movies for our new user above.

让我们看看为新用户预测评分最高的电影。回顾一下，该用户向量偏好喜剧片（Comedy）和爱情片（Romance）。
下面，我们将使用一组电影/物品向量 `item_vecs`，它为训练集/测试集中的每部电影都包含一个向量。将其与上面的用户向量匹配，并使用缩放后的向量为上述新用户预测所有电影的评分。

In [ ]:
# generate and replicate the user vector to match the number movies in the data set.
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# scale the vectors and make predictions for all movies. Return results sorted by rating.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, scalerUser, scalerItem, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

If you do create a user above, it is worth noting that the network was trained to predict a user rating given a user vector that includes a **set** of user genre ratings.  Simply providing a maximum rating for a single genre and minimum ratings for the rest may not be meaningful to the network if there were no users with similar sets of ratings.

如果你确实在上面创建了自定义用户，值得注意的是，该网络是在给定包含**一组**用户类型评分的用户向量的情况下训练以预测用户评分的。如果仅对单个类型给出最高评分而对其他类型给出最低评分，并且训练集中没有具有类似评分组合的用户，那么这对网络来说可能没有意义。

#### Predictions for an existing user.
Let's look at the predictions for "user 36", one of the users in the data set. We can compare the predicted ratings with the model's ratings. Note that movies with multiple genre's show up multiple times in the training data. For example,'The Time Machine' has three genre's: Adventure, Action, Sci-Fi

#### 为现有用户进行预测
让我们看看对数据集中一位用户“user 36”的预测。我们可以将预测评分与模型的实际评分进行比较。请注意，具有多种类型的电影在训练数据中会出现多次。例如，“The Time Machine”具有三种类型：冒险（Adventure）、动作（Action）、科幻（Sci-Fi）。

In [ ]:
uid =  36 
# form a set of user vectors. This is the same vector, transformed and repeated.
user_vecs, y_vecs = get_user_vecs(uid, scalerUser.inverse_transform(user_train), item_vecs, user_to_genre)

# scale the vectors and make predictions for all movies. Return results sorted by rating.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, 
                                                                      scalerUser, scalerItem, scaledata=scaledata)
sorted_y = y_vecs[sorted_index]

#print sorted predictions
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, item_features, ivs, uvs, movie_dict, maxcount = 10)

#### Finding Similar Items
The neural network above produces two feature vectors, a user feature vector $v_u$, and a movie feature vector, $v_m$. These are 32 entry vectors whose values are difficult to interpret. However, similar items will have similar vectors. This information can be used to make recommendations. For example, if a user has rated "Toy Story 3" highly, one could recommend similar movies by selecting movies with similar movie feature vectors.

A similarity measure is the squared distance between the two vectors $ \mathbf{v_m^{(k)}}$ and $\mathbf{v_m^{(i)}}$ :
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

#### 查找相似物品
上述神经网络生成两个特征向量：用户特征向量 $v_u$ 和电影特征向量 $v_m$。它们是 32 维的向量，其数值难以直接解释。然而，相似的物品会具有相似的向量。这一信息可用于生成推荐。例如，如果用户对“Toy Story 3”给出了很高的评分，可以通过选择具有相似电影特征向量的电影来推荐相似的电影。

一种相似度度量是两个向量 $\mathbf{v_m^{(k)}}$ 和 $\mathbf{v_m^{(i)}}$ 之间的平方距离：
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$

<a name="ex01"></a>
### Exercise 1

Write a function to compute the square distance.

### 练习 1

编写一个计算平方距离的函数。

In [ ]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    ### START CODE HERE ###     
    
    ### END CODE HERE ###     
    return (d)

In [ ]:
# Public tests
test_sq_dist(sq_dist)

In [ ]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
  While a summation is often an indication a for loop should be used, here the subtraction can be element-wise in one statement. Further, you can utilized np.square to square, element-wise, the result of the subtraction. np.sum can be used to sum the squared elements.
    
</details>

    


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
  虽然求和通常意味着应该使用 for 循环，但在这里，减法可以在一条语句中按元素完成。此外，你可以使用 `np.square` 对减法结果按元素求平方，并使用 `np.sum` 对平方后的元素求和。
    
</details>

A matrix of distances between movies can be computed once when the model is trained and then reused for new recommendations without retraining. The first step, once a model is trained, is to obtain the movie feature vector, $v_m$, for each of the movies. To do this, we will use the trained `item_NN` and build a small model to allow us to run the movie vectors through it to generate $v_m$.

电影之间的距离矩阵可以在模型训练完成后计算一次，然后无需重新训练即可复用于新的推荐。模型训练完成后的第一步是获取每部电影的电影特征向量 $v_m$。为此，我们将使用训练好的 `item_NN` 并构建一个小型模型，以便将电影向量输入其中来生成 $v_m$。

In [ ]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features))    # input layer
vm_m = item_NN(input_item_m)                                       # use the trained item_NN
vm_m = tf.linalg.l2_normalize(vm_m, axis=1)                        # incorporate normalization as was done in the original model
model_m = Model(input_item_m, vm_m)                                
model_m.summary()

Once you have a movie model, you can create a set of movie feature vectors by using the model to predict using a set of item/movie vectors as input. `item_vecs` is a set of all of the movie vectors. Recall that the same movie will appear as a separate vector for each of its genres. It must be scaled to use with the trained model. The result of the prediction is a 32 entry feature vector for each movie.

一旦有了电影模型，就可以将一组物品/电影向量作为输入，使用该模型进行预测，从而创建一组电影特征向量。`item_vecs` 是所有电影向量的集合。回顾一下，同一部电影会为其每种类型分别表示为一个独立的向量。在输入到训练好的模型之前，必须先对其进行缩放。预测结果是每部电影对应的一个 32 维特征向量。

In [ ]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

Let's now compute a matrix of the squared distance between each movie feature vector and all other movie feature vectors:
<figure>
    <left> <img src="./images/distmatrix.PNG"   style="width:400px;height:225px;" ></center>
</figure>

现在让我们计算每部电影的特征向量与所有其他电影特征向量之间的平方距离矩阵：

We can then find the closest movie by finding the minimum along each row. We will make use of [numpy masked arrays](https://numpy.org/doc/1.21/user/tutorial-ma.html) to avoid selecting the same movie. The masked values along the diagonal won't be included in the computation.

然后我们可以通过找到每一行的最小值来找出最接近的电影。我们将使用 [numpy 掩码数组（masked arrays）](https://numpy.org/doc/1.21/user/tutorial-ma.html) 来避免选中同一部电影。对角线上的被掩码值将不参与计算。

In [ ]:
count = 50
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

The results show the model will suggest a movie from the same genre.

结果表明，模型会推荐来自同一类型的电影。

<a name="4"></a>
## 4 - Congratulations! <img align="left" src="./images/film_award.png" style=" width:40px;">
You have completed a content-based recommender system.    

This structure is the basis of many commercial recommender systems. The user content can be greatly expanded to incorporate more information about the user if it is available.  Items are not limited to movies. This can be used to recommend any item, books, cars or items that are similar to an item in your 'shopping cart'.

## 4 - 恭喜！
你已经完成了一个基于内容的推荐系统。

这种结构是许多商业推荐系统的基础。如果有更多关于用户的信息，用户内容可以大幅扩展以纳入这些信息。物品也不限于电影——它可以用于推荐任何物品，例如书籍、汽车，或者与你“购物车”中商品相似的物品。